In [8]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests
from langchain.agents import create_react_agent, AgentExecutor


In [9]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [10]:
search_tool = TavilySearchResults(max_results=2)

In [11]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [12]:
result = search_tool.invoke("whats the latest news of usa mid term election")
result

[{'url': 'https://www.nbcnews.com/politics/2026-election',
  'content': "## Democrat Adam Hamilton pulls off upset over GOP Sen. Roger Marshall in Kansas\n\n## Democrat Sherrod Brown unseats Sen. Jon Husted in Ohio\n\n## Roy Cooper flips North Carolina for Democrats\n\n## GOP Sen. Ashley Moody wins Florida Senate special election\n\n## Governor\n\n## Illinois Gov. JB Pritzker rolls out a Day One road map for Congress\n\n## Gina Hinojosa to become Texas’ first Hispanic governor after stunning win\n\n## Democrat Rob Sand flips control of the Iowa governorship\n\n## Record number of female governors elected\n\n## Amy Acton defeats Republican Vivek Ramaswamy in Ohio\n\n## More results\n\n## GOP Rep. Max Miller loses re-election in Ohio following abuse allegations from ex-wife\n\n## Lawmaker who led GOP efforts to keep the House loses re-election bid [...] Five data takeaways showing how Democrats wrested back control of the House and the Senate.\n\n8h ago\n\n## Live Updates\n\n## Live Upda

In [13]:

llm = ChatOpenAI(
    model="gemini-3.5-flash",
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


In [14]:
response = llm.invoke("Whats your model name?")
response

AIMessage(content='I am Gemini 3.6 Flash, a large language model built by Google.', response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 6, 'total_tokens': 102, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'gemini-3.5-flash', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-46014739-dffd-4609-8174-a011395d576c-0')

In [15]:
prompt = hub.pull("hwchase17/react")

g:\GITHUB AND PROJECTS\Single Agent\venv\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [16]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [17]:
tools = [search_tool, get_weather_data]

In [18]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [19]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True

)

In [20]:
response = agent_executor.invoke({
    "input": (
        "Find the capital of bangladesh"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...
Action: get_weather_data
Action Input: {"city": "Dhaka"}City: {"city": "Dhaka"}
Temperature: 31°C
Weather: Sunny
Humidity: 50%Thought: I now know the capital of Bangladesh is Dhaka and I have retrieved its current weather information.

Final Answer: The capital of Bangladesh is Dhaka. The current weather in Dhaka is Sunny with a temperature of 31°C and 50% humidity.

> Finished chain.


In [21]:
print(response["output"])

The capital of Bangladesh is Dhaka. The current weather in Dhaka is Sunny with a temperature of 31°C and 50% humidity.
